# Transformers

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
import lightning as L
from torch.utils.data import TensorDataset, DataLoader

## Create the Datasets that we will use.

We will translate **Let's go** and **to go** into spanish. **Let's go** should translate to **vamos<EOS>** and **to go** should translate to **ir<EOS>**

In [2]:
input_vocab = {'<SOS>': 0,
               'lets': 1,
               'to': 2,
               'go': 3}

output_vocab = {'<SOS>': 0,
                'ir': 1,
                'vamos': 2,
                'y': 3,
                '<EOS>':4}

inputs = torch.tensor([[1, 3],
                       [2, 3]])

labels = torch.tensor([[2],
                       [1]])

dataset = TensorDataset(inputs, labels)
dataloader = DataLoader(dataset)

## Position Encoding

Helps the transformer keep track of the order of the words in the input and the output.
To do that we use a series of alternating sine and cosine curves.
Since the position encoding values never change (the first token always uses the same position encoding values regardless of what that token is) we can precompute them and save them in a lookup table (that makes adding position encoding values super fast). 

**The PositionEncoding class takes word embeddings and adds a unique positional signal to each one, so the transformer knows the order of the tokens in the sequence.**

In [3]:
class PositionEncoding(nn.Module):
    def __init__(self, d_model=2, max_len=3):
        # d_model: The dimension of the transformer (also the number of embedding values per token)
        #           In 'Attention Is All You Need' d_model was set as d_model=512
        # max_len: maximum number of tokens we allow as input. We will use only 2 word phrases + <SOS>, so we set max_len=3

        super().__init__()

        # We create the lookup table of position encoding values and initialize all of them to 0
        # To do that we create a matric of 0s that has max_len rows and d_model columns. 
        # A lookup table is just a pre-computed table where you say "give me the row for position 2" and you get back [..., ...]. 
        # No calculation needed at that point — it was all computed once upfront when you built the table.
        pe = torch.zeros(max_len, d_model)

        # We create a sequence of numbers for each position that a token can have in the input or output. 
        # We change them to floats and we use .unsqueeze(1) to convert a list of numbers into a matrix with one row for each index and all indices in a single column.
        # In this example with max_len = 3 we create a matrix with 3 rows and 1 column.
        position = torch.arange(start=0, end=max_len, step=1).float().unsqueeze(1) # This gives us [[0], [1], [2]]  (a column vector of shape (3, 1).)


        # Now we determine the y-axis coordinates on the sine and cosine curves
        div_term = 1/torch.tensor(10000.0)**(torch.arange(start=0, end=d_model, step=2).float()/d_model)

        # We calculate the actual positional encoding values (pe was initialized as a tensor of 0s)
        pe[:, 0::2] = torch.sin(position * div_term) # Every other column, starting with the 1st, has sin() values
        pe[:, 1::2] = torch.cos(position * div_term) # Every other column, starting with the 2nd, has cos() values
        # In the code above, before the ',' is the row and by using ':' we select all rows. After the ',' is the column
        # and by using 0::2 we say 'start from column 0 and go to the end with a stepsize of 2



        # Next we register pe. 
        # A PyTorch model as having two kinds of numbers stored inside it:
        # 1) Trainable parameters — weights and biases that the optimizer adjusts during training (e.g. attention weights). These live in model.parameters().
        # 2) Buffers — fixed numbers the model needs but should never change during training.
        # pe belongs in the second category. The positional encoding values are pure math — sine and cosine — not something the model should be tweaking. 
        # So we use register_buffer to say:
        # "Store this inside the model, but keep it away from the optimizer."
        self.register_buffer('pe', pe) # Using 'register_buffer()' prevents 'pe' from getting passed to the optimizer

    # The forward method is what is called by default when we use a PositionEncoding() object. 
    # When we create a PositionEncoding() object, "pe=PositionEncoding()", we can pass position encoding values to the word embeddings with pe(word_embeddings)
    def forward(self, x):
        # x = word embedding values
        # x.size(0) returns the number of tokens in the input sequence
        return x + self.pe[:x.size(0), :] 

## Attention

We will code the 'Attention' class to do all of the types of attention that a transformer might need: **Self-Attention, Masked Self-Attention** (this one is used by the Decoder during training) and **Encoder-Decoder Attention**

**Self Attention** is the type of attention used in Encoder-Decoder transformers and Encoder-Only Transformers and allows every word in a phrase to define a relationship with any other word in the phrase (regardless of the order of the words). 

**Masked Self-Attention** is used by all types of transformers and it allows each word in a phrase to define a relationship with itself and the words that came before it.

**Encoder-Decoder Attention** is only used in Encoder-Decoder transformers, where there is a distinct seperation of the part of the transformer that processes in the input (the encoder) from the part that generates the output (the decoder). Encoder-Decoder Attention lets each word in the output (the decoder) to define relationships with all the words in the input (in the encoder)

**The Attention class takes input representations, projects them into queries/keys/values, computes how much each token should attend to every other token, and returns a new representation for each token that blends information from the whole sequence.**

In [4]:
class Attention(nn.Module):
    def __init__(self, d_model=2):
        super().__init__()
        # We initialize the Weights we will use to create the query (q), key (k), and value (v) for each token
        # Most implementations include bias terms but we dont include them as they werent part of the original Attention is All You Need paper
        self.W_q = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_k = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_v = nn.Linear(in_features=d_model, out_features=d_model, bias=False)

        # We set those here for simplicity down the road
        self.row_dim = 0 # row dimension is 0
        self.col_dim = 1 # column dimension is 1
    
    def forward(self, encodings_for_q, encodings_for_k, encodings_for_v, mask=None):
        # We create the Query, Key, and Values using the encodings associated with each token 
        # For normal Self-Attention and Masked Self-Attention encodings_for_q == encodings_for_k == encodings_for_v
        # For Encoder-Decoder Attention, encodings_for_q comes from the Decoder and encodings_for_k and encodings_for_v are different and come from the Encoder
        q = self.W_q(encodings_for_q)
        k = self.W_k(encodings_for_k)
        v = self.W_v(encodings_for_v)

        # We compute the raw attention scores using the equation (q * K^T) / sqrt(d_model)
        sims = torch.matmul(q, k.transpose(dim0=self.row_dim, dim1=self.col_dim))
        scaled_sims = sims / torch.tensor(q.size(self.col_dim)**0.5)

        if mask is not None:
            # We replace things we want masked out with a bery large negative number (to approximate -infinity) so that
            # Softmax() will give all masked elements an output value of 0
            '''
            `scaled_sims` is a `(seq_len, seq_len)` matrix where each cell `[i, j]` means "how much should token i attend to token j?"
            For a 3-token sequence it looks like this:

                        <SOS>   cat   sat
            <SOS>        [ 0.9,  0.3,  0.8 ]   ← how much <SOS> attends to each token
            cat          [ 0.5,  0.7,  0.2 ]   ← how much "cat" attends to each token
            sat          [ 0.1,  0.4,  0.6 ]   ← how much "sat" attends to each token
           

            In the decoder, token `i` is not allowed to look at tokens that come **after** it — that would be peeking at future words the model hasn't generated yet. So we block out the upper triangle:
       
                        <SOS>   cat   sat
            <SOS>        [ 0.9,  -1e9, -1e9 ]   ← <SOS> can only see itself
            cat          [ 0.5,  0.7,  -1e9 ]   ← "cat" can see <SOS> and itself
            sat          [ 0.1,  0.4,  0.6  ]   ← "sat" can see everything
            When softmax runs on a row, -1e9 becomes essentially 0 — those positions are completely ignored. That's the mask.
            '''
            scaled_sims = scaled_sims.masked_fill(mask=mask, value=-1e9)

        # Next we apply Softmax to determine what percent of each token's value to use in the final attention values
        attention_percents = F.softmax(scaled_sims, dim=self.col_dim) # Softmax converts the raw scores into percentages that sum to 1

        # Scale the values by their associated percentages and add them up
        attention_scores = torch.matmul(attention_percents, v)
        '''
        After softmax, `attention_percents` looks something like this (each row sums to 1):

                    <SOS>   cat   sat
        <SOS>        [ 1.0,   0.0,  0.0 ]
        cat          [ 0.4,   0.6,  0.0 ]
        sat          [ 0.1,   0.3,  0.6 ]


        And `v` (the value matrix) looks like:

                dim0   dim1
        <SOS>  [ 0.2,   0.5 ]
        cat    [ 0.9,   0.1 ]
        sat    [ 0.4,   0.8 ]


        The matrix multiplication blends the value vectors using those percentages. For "cat" for example:

        cat output = 0.4 × [0.2, 0.5]   (40% of <SOS>'s value)
                + 0.6 × [0.9, 0.1]   (60% of cat's own value)
        
        each token's output is a blend of information from the tokens it attended to, weighted by how much attention it paid to each one.
        That blended vector is what gets passed forward in the network.

        '''

        return attention_scores



## Encoder 

A basic Encoder simply brings together
* Word Embedding
* Position Encoding
* Self-Attention
* Residual Connections

The encoder's job is to read and understand the input sequence.
It takes the input tokens and produces a rich representation for each token that captures two things:

What the token means — from the word embedding
How it relates to every other token in the sequence — from self-attention

In [5]:
class Encoder(nn.Module):
    def __init__(self, num_tokens=4, d_model=2, max_len=3):
        super().__init__()
        L.seed_everything(seed=42)

        # For this example we will use a 'single layer' encoder. 
        # If we wanted to add multiple layers, we would take the output of one encoder and use it as input to the next one
        # This is the word embedding layer
        self.we = nn.Embedding(num_embeddings=num_tokens,
                               embedding_dim=d_model)
        # This is the positional encoding class 
        self.pe = PositionEncoding(d_model=d_model,
                                   max_len=max_len)
        # This is the attention class
        self.self_attention = Attention(d_model=d_model)
        # If we wanted to do multi-head Attention we could initialize more Attention objects  like this...
        # self.self_attention2 = Attention(d_model=d_mode)
        # self.self_attention3 = Attention(d_model=d_model)

        # If d_model=2 then using 3 self_attention objects would result in d_model * 3 = 6 self-attention values per token,
        # so we would need to initialize a matrix of weights to reduce the dimensions of the self attention values back down to d_model 
        # self.reduce_attention_dim = nn.Linear(in_features=(num_attention_heads * d_model), out_features=d_model)

    def forward(self, token_ids):
        # We get the word embeddings
        word_embeddings = self.we(token_ids) # convert raw token IDs into embedding vectors

        # We add positional encoding to the word embeddings
        position_encoded = self.pe(word_embeddings)

        # We calculate the self attention values
        self_attention_values = self.self_attention(position_encoded, # query
                                                    position_encoded, # key
                                                    position_encoded) # value
        # The above runs self-attention by passing position_encoded as all three arguments — query, key and value are all the same. 
        # This is what makes it self-attention: every token attends to every other token in the same sequence. 
        # No mask here because the encoder is allowed to look at the full input.
        
        # If we were doing multi-head attention, we would calculate the self-attention values with the other attention objects like this...
        # self_attention_values_2 = self.self_attention_2()
        # self_attention_values_3 = self.self_attention_3()
        # Then we would concatenate all the self attention values -> all_self_attention_Values = torch.cat(self_attentoon_values_1, ...)
        # and run them through reduce_dim to get back to d_model values per token
        # final_self_attention_values = self.reduce_attention_dim(all_self_attention_values)

        # We add the position encoded values to the self attention values to get the output values
        output_values = position_encoded + self_attention_values # This is a residual connection — we add the input of the attention layer back to its output.
                                                                 # It prevents information loss and helps gradients flow during backpropagation.
        return output_values

## Decoder 

The decoder class is almost the same as the encoder, except that it also included Encoder-Decoder Attention (where the 'Keys' and 'Values' are created from the outputs from the Encoder), a fully connected layer so that we can have 5 outputs (one per word in the vocabulary), and a SoftMax() function to select the output token.

A basic Decoder simply brings together:

* Word Embedding
* Position Encoding
* Self-Attention
* Residual Connections
* Encoder-Decoder Attention
* A fully connected layer
* SoftMax (The loss function nn.CrossEntropyLoss() applies the Softmax automatically)

The decoder's job is to generate the output sequence one token at a time.
It differs from the encoder in three key ways:

* Masked self-attention — the decoder can only look at tokens it has already generated, not future ones. This is because during training we feed the entire output sequence at once, so we need the mask to prevent cheating.
* Encoder-decoder attention — after masked self-attention, the decoder attends to the encoder's output. This is how the decoder "reads" the input sequence while generating each output token. The decoder asks "given what I've generated so far, which parts of the input should I focus on?"
* Final linear layer — converts the attention values into a score for each token in the vocabulary, so we can pick the most likely next token.

In [6]:
class Decoder(nn.Module):
    def __init__(self, num_tokens=4, d_model=2, max_len=3):
        super().__init__()
        L.seed_everything(seed=43) # We use different seed number from the encoder so that we start out with different embedding values 

        # Like in the encoder, we are using a single layer decoder
        self.we = nn.Embedding(num_embeddings=num_tokens,
                               embedding_dim=d_model)
        self.pe = PositionEncoding(d_model=d_model,
                                   max_len=max_len)
        
        # Like in the encoder, we are using a single head for the self-attention and encoder-decoder attention
        # Two attention objects this time. The first is for masked self-attention (decoder attending to itself), 
        # the second is for encoder-decoder attention (decoder attending to the encoder output).
        self.self_attention = Attention(d_model=d_model)
        self.enc_dec_attention = Attention(d_model=d_model)

        #A fully connected layer at the end. 
        # In a real transformer this would have out_features=num_tokens — outputting one score per vocabulary token so you can pick the most likely next word. 
        # Here it's kept at d_model to keep the example simple.
        self.fc_layer = nn.Linear(in_features=d_model, out_features=num_tokens)
        
        self.row_dim = 0
        self.col_dim = 1

    def forward(self, token_ids, encoder_values): # encoder_values is the output of the encoder — the context-aware representations of the input sequence.
        word_embeddings = self.we(token_ids)
        position_encoded = self.pe(word_embeddings)

        # For the decoder we need to use "masked self-attention" so that 
        # when we train the decoder cant look ahead at what words come after the current word it's working on. 
        '''
        

        `torch.ones(3, 3)` creates a matrix of all 1s:
       
        [[1, 1, 1],
        [1, 1, 1],
        [1, 1, 1]]

        `torch.tril()` keeps only the lower triangle:

        [[1, 0, 0],
        [1, 1, 0],
        [1, 1, 1]]

        `mask == 0` flips it to a boolean mask where `True` means "block this position":

        [[False, True,  True ],
        [False, False, True ],
        [False, False, False]]

        This gets passed to the Attention class where True positions get filled with -1e9 and become zero after softmax.
        '''
        mask = torch.tril(torch.ones((token_ids.size(dim=self.row_dim), token_ids.size(dim=self.row_dim))))
        mask = mask == 0

        # This is the Masked self-attention
        self_attention_values = self.self_attention(position_encoded, # Query comes from the decoder
                                                    position_encoded, # Key comes from the decoder
                                                    position_encoded, # Value comes from the decoder
                                                    mask=mask)
        # residual connection
        residual_connection_values = position_encoded + self_attention_values 

        # This is the encoder-decoder attention
        enc_dec_attention_values = self.enc_dec_attention(residual_connection_values, # Query comes from the decoder (residual_connection_values) — the decoder is asking "what do I need from the input?"
                                                          encoder_values, # Key and Value come from the encoder (encoder_values) — the encoder is providing the information to attend to
                                                          encoder_values) 
        # residual connection
        residual_connection_values = enc_dec_attention_values + residual_connection_values


        fc_layer_output = self.fc_layer(residual_connection_values)
        # We are not passing the fc_layer_output to a SoftMax because the loss function we are using will apply it for us
        return fc_layer_output

## Transformer

The Transformer() class connects the outputs from the Encoder to the Decoder 

In [7]:
class Transformer(L.LightningModule):
    def __init__(self, input_size, output_size, d_model=2, max_len=3):
        super().__init__()

        self.encoder = Encoder(num_tokens=len(input_vocab), d_model=d_model, max_len=max_len) # we add the Encoder
        self.decoder = Decoder(num_tokens=len(output_vocab), d_model=d_model, max_len=max_len) # we add the Decoder

        self.loss = nn.CrossEntropyLoss()

    def forward(self, inputs, labels):
        encoder_values = self.encoder(inputs) # We run the encoder first
        output_presoftmax = self.decoder(labels, encoder_values) # Wr pass the output of the encoder to the decoder along with the labels
        return (output_presoftmax)
    
    def configure_optimizers(self):
        return Adam(self.parameters(), lr=0.1)
    
    def training_step(self, batch, batch_idx):
        # We take a batch of input/output pairs and prepares them by adding <SOS> and <EOS> tokens in the right places.
        input_i, label_i = batch 

        # We append the <SOS> token to the tokens used as input to the Encoder
        input_tokens = torch.cat((torch.tensor([0]), input_i[0]))
        # Then we do the same for the tokens used as input to the Decoder
        teacher_forcing = torch.cat((torch.tensor([0]), label_i[0]))

        # Next we add <EOS> token to the end of the known output
        expected_output = torch.cat((label_i[0], torch.tensor([4])))
        
        # We call the forward method, which runs input_tokens through the encoder and teacher_forcing through the decoder and returns the predictions.
        output_i = self.forward(input_tokens, teacher_forcing)

        # Lightning takes the returned loss and calls .backward() to compute gradients throughout the entire model via backpropagation.
        # The Adam optimizer then uses those gradients to update all the trainable weights, and the cycle repeats for the next batch.
        loss = self.loss(output_i, expected_output)
        return loss
        

In [8]:
max_length = 3
transformer = Transformer(len(input_vocab), len(output_vocab), d_model=2, max_len=max_length)

# Runs the encoder on the input sequence [0, 1, 3] which represents <SOS>, let's, go.
encoder_values = transformer.encoder(torch.tensor([0, 1, 3])) # expected output: 0, 2, 4 that corresponds to <SOS> vamos <EOS>

# We  set the first predicted token to <SOS> to initialize the decoder
predicted_ids = torch.tensor([0]) # We're telling the decoder "start generating now"
for i in range(max_length): # We cap it at max_length as a safety measure in case the model never predicts <EOS>

    # We run the decoder with whatever tokens have been predicted so far, plus the fixed encoder values. 
    # Each iteration `predicted_ids` grows by one token, so the decoder has more context each time:
    # iteration 1: decoder sees [<SOS>]
    # iteration 2: decoder sees [<SOS>, vamos]
    # iteration 3: decoder sees [<SOS>, vamos, <EOS>]
    prediction = transformer.decoder(predicted_ids, encoder_values)

    # Next
    # - prediction[-1, :] — takes the last row of the prediction matrix.
    #  Since we only want to predict the next token, we only care about the output of the last position
    # - torch.argmax() picks the index of the highest score, which corresponds to the most likely next token
    predicted_id = torch.tensor([torch.argmax(prediction[-1,:])])

    # We append the newly predicted token id to the list of predicted ids. 
    predicted_ids = torch.cat((predicted_ids, predicted_id))

    if (predicted_id == 4): # If the prediction is <EOS> (Token 4 is <EOS>) then we are done
        break

print("\npredicted_ids: ", predicted_ids)

Seed set to 42
Seed set to 43



predicted_ids:  tensor([0, 2, 0, 1])


## Train the Transformer

In [9]:
transformer = Transformer(len(input_vocab), len(output_vocab), d_model=2, max_len=3)
trainer = L.Trainer(max_epochs=30, accelerator="cpu")
trainer.fit(transformer, train_dataloaders=dataloader)

Seed set to 42
Seed set to 43
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/odysseasgeorgiades/Desktop/Neural Networks/.venv/lib/python3.14/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name    | Type             | Params | Mode  | FLOPs
-------------------------------------------------------------
0 | encoder | Encoder          | 20     | train | 0    
1 | decoder | Decoder          | 49     | train | 0    
2 | loss    | Cro

Epoch 29: 100%|██████████| 2/2 [00:00<00:00, 638.60it/s, v_num=3]

`Trainer.fit` stopped: `max_epochs=30` reached.


Epoch 29: 100%|██████████| 2/2 [00:00<00:00, 384.32it/s, v_num=3]


## Use the Transformer

In [10]:
max_length = 3
row_dim=0
col_dim=1

encoder_values = transformer.encoder(torch.tensor([0, 1, 3])) # expected output: 0, 2, 4 that corresponds to <SOS> vamos <EOS>

predicted_ids = torch.tensor([0]) 
for i in range(max_length):
    prediction = transformer.decoder(predicted_ids, encoder_values)

    predicted_id = torch.tensor([torch.argmax(prediction[-1,:])])

    predicted_ids = torch.cat((predicted_ids, predicted_id))

    if (predicted_id == 4):
        break

print("\npredicted_ids: ", predicted_ids)


predicted_ids:  tensor([0, 2, 4])


Correctly predicted [0, 2, 4] which corresponds to < SOS > vamos < EOS >